HOW TO RUN NEO4J:

https://AWS-PUBLIC-IP:7473
    Example: https://3.12.45.67:7473

Use login credintials:
    Username: neo4j
    Password: ucb_mids_w205

In [1]:
import neo4j
import pandas as pd
from IPython.display import display
import numpy as np
import requests
from io import StringIO


In [2]:
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-22/population.csv"
response = requests.get(url, verify=False)
population = pd.read_csv(StringIO(response.text))
print("", population.shape)
population.head()

/opt/conda/lib/python3.9/site-packages/urllib3/connectionpool.py:1013: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


 (64809, 16)


,year,coo_name,coo,coo_iso,coa_name,coa,coa_iso,refugees,asylum_seekers,returned_refugees,idps,returned_idps,stateless,ooc,oip,hst
0,2010,Afghanistan,AFG,AFG,Afghanistan,AFG,AFG,0,0,0,351907,3366,0,838250,NaN,NaN
1,2010,Iran (Islamic Rep. of),IRN,IRN,Afghanistan,AFG,AFG,30,21,0,0,0,0,0,NaN,NaN
2,2010,Iraq,IRQ,IRQ,Afghanistan,AFG,AFG,6,0,0,0,0,0,0,NaN,NaN
3,2010,Pakistan,PAK,PAK,Afghanistan,AFG,AFG,6398,9,0,0,0,0,0,NaN,NaN
4,2010,Egypt,ARE,EGY,Albania,ALB,ALB,5,0,0,0,0,0,0,NaN,NaN


## Driver & Session for Neo4j Database

In [3]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [4]:
session = driver.session(database="neo4j")

## Wipe out all Nodes and Relationships

In [5]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

## Put Results of Cypher Query in Pandas Dataframe

In [6]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [7]:
query = """
UNWIND range(1,10) AS id
CREATE (:Node {id:id})
"""

session.run(query)

In [8]:
df = my_neo4j_run_query_pandas("""
MATCH (n:Node)
RETURN n.id
""")

display(df)

,n.id
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


## Creating the Database


In [9]:
# Step 1: Wipe the database
my_neo4j_wipe_out_database()

# Step 2: Create unified Country nodes (one per country, regardless of role)
all_countries = pd.concat([
    population[["coo_name", "coo_iso"]].rename(columns={"coo_name": "name", "coo_iso": "iso"}),
    population[["coa_name", "coa_iso"]].rename(columns={"coa_name": "name", "coa_iso": "iso"})
]).drop_duplicates(subset=["name"]).to_dict("records")

query_countries = """
UNWIND $rows AS row
MERGE (c:Country {name: row.name})
SET c.iso = row.iso
"""
session.run(query_countries, rows=all_countries)

# Step 3: Create Year nodes
year_rows = population[["year"]].drop_duplicates().to_dict("records")
query_year = """
UNWIND $rows AS row
MERGE (y:Year {year: row.year})
"""
session.run(query_year, rows=year_rows)

# Step 4: Create REFUGEES_TO relationships — one per origin/asylum/year combo
population_rows = population[[
    "coo_name", "coa_name", "year",
    "refugees", "asylum_seekers", "returned_refugees",
    "idps", "returned_idps", "stateless", "ooc", "hst"
]].fillna(0).to_dict("records")

query_relationships = """
UNWIND $rows AS row
MATCH (origin:Country {name: row.coo_name})
MATCH (asylum:Country {name: row.coa_name})
MATCH (y:Year {year: row.year})
MERGE (origin)-[r:REFUGEES_TO {year: row.year}]->(asylum)
SET r.refugees        = row.refugees,
    r.asylum_seekers  = row.asylum_seekers,
    r.returned_refugees = row.returned_refugees,
    r.idps            = row.idps,
    r.returned_idps   = row.returned_idps,
    r.stateless       = row.stateless,
    r.ooc             = row.ooc,
    r.hst             = row.hst
MERGE (origin)-[:IN_YEAR]->(y)
MERGE (asylum)-[:IN_YEAR]->(y)
"""
session.run(query_relationships, rows=population_rows)

# Step 5: Verify
df_countries = my_neo4j_run_query_pandas("MATCH (c:Country) RETURN c.name AS country, c.iso AS iso LIMIT 10")
df_years     = my_neo4j_run_query_pandas("MATCH (y:Year) RETURN y.year AS year ORDER BY y.year LIMIT 10")
df_rels      = my_neo4j_run_query_pandas("""
    MATCH (o:Country)-[r:REFUGEES_TO]->(a:Country)
    RETURN o.name AS origin, a.name AS asylum, r.year AS year, r.refugees AS refugees
    ORDER BY r.refugees DESC LIMIT 10
""")

display(df_countries)
display(df_years)
display(df_rels)


,country,iso
0,Cabo Verde,CPV
1,New Zealand,NZL
2,"China, Macao SAR",MAC
3,Kiribati,KIR
4,Austria,AUT
5,Andorra,AND
6,Finland,FIN
7,Iceland,ISL
8,Malta,MLT
9,Bermuda,BMU


,year
0,2010
1,2011
2,2012
3,2013
4,2014
5,2015
6,2016
7,2017
8,2018
9,2019


,origin,asylum,year,refugees
0,Syrian Arab Rep.,Türkiye,2021,3737369
1,Syrian Arab Rep.,Türkiye,2020,3641370
2,Syrian Arab Rep.,Türkiye,2018,3622366
3,Syrian Arab Rep.,Türkiye,2019,3576370
4,Syrian Arab Rep.,Türkiye,2022,3535898
5,Syrian Arab Rep.,Türkiye,2017,3424237
6,Afghanistan,Iran (Islamic Rep. of),2022,3413249
7,Syrian Arab Rep.,Türkiye,2016,2823987
8,Syrian Arab Rep.,Türkiye,2015,2503549
9,Afghanistan,Pakistan,2010,1899842


# Checking GDS version

In [10]:
# Check GDS version
query = "RETURN gds.version() AS version"
df = my_neo4j_run_query_pandas(query)
display(df)

,version
0,2.5.7


## Shortest Path Graph - Both Directions

In [13]:
# Step 1: Drop stale projections
for name in ['refugee-graph', 'refugee-graph-reverse']:
    session.run(f"""
        CALL gds.graph.exists('{name}') YIELD exists
        WITH exists WHERE exists = true
        CALL gds.graph.drop('{name}') YIELD graphName
        RETURN graphName
    """)

# Step 2: Create Graph 1 (Origin -> Asylum)
session.run("""
CALL gds.graph.project(
    'refugee-graph',
    'Country',
    {
        REFUGEES_TO: {
            orientation: 'NATURAL',
            properties: ['refugees']
        }
    }
)
""")

# Step 3: Create Graph 2 (Asylum -> Origin)
session.run("""
CALL gds.graph.project(
    'refugee-graph-reverse',
    'Country',
    {
        REFUGEES_TO: {
            orientation: 'REVERSE',
            properties: ['refugees']
        }
    }
)
""")



In [ ]:
'''

Understanding each graph:

Graph 1 — Origin → Asylum (natural direction)

MATCH (origin:Country)-[r:REFUGEES_TO]->(asylum:Country)
RETURN origin, r, asylum
LIMIT 15


Graph 2 — Asylum → Origin (reverse direction)
MATCH (origin:Country)-[r:REFUGEES_TO]->(asylum:Country)
RETURN asylum, r, origin
LIMIT 15


Example prompts in Neo4j browser:

-- Graph 1: Where do Syrian refugees go?

MATCH (origin:Country {name: "Syrian Arab Rep."})-[r:REFUGEES_TO]->(asylum:Country)
RETURN origin, r, asylum

-- Graph 2: Who sends refugees to Germany?

MATCH (origin:Country)-[r:REFUGEES_TO]->(asylum:Country {name: "Germany"})
RETURN origin, r, asylum

'''


###

## Betweenness Centraility - Transit Hubs

In [14]:
# Step 1: Now run betweenness
session.run("""
CALL gds.betweenness.write('refugee-graph', {
    writeProperty: 'betweenness'
})
""")

df_betweenness = my_neo4j_run_query_pandas("""
MATCH (c:Country)
WHERE c.betweenness > 0
RETURN c.name AS country, c.iso AS iso, c.betweenness AS betweenness_score
ORDER BY betweenness_score DESC
LIMIT 20
""")
display(df_betweenness)

,country,iso,betweenness_score
0,United States of America,USA,9016.857306
1,Russian Federation,RUS,2739.240095
2,Lebanon,LBN,2419.060225
3,Syrian Arab Rep.,SYR,2087.199464
4,Venezuela (Bolivarian Republic of),VEN,1988.791619
5,Burundi,BDI,1870.800114
6,Türkiye,TUR,1633.003128
7,Canada,CAN,1495.926704
8,Iraq,IRQ,1138.053969
9,Sudan,SDN,1086.865878


In [ ]:
'''
Understanding the graph, do this to see all countries and thier scores:

MATCH (c:Country)
WHERE c.betweenness > 0
RETURN c
ORDER BY c.betweenness DESC
LIMIT 10


Example prompt for Neo4j browser:

MATCH (origin:Country)-[r:REFUGEES_TO]->(asylum:Country)
WHERE origin.betweenness > 500
RETURN origin, r, asylum
LIMIT 10

'''